# 16 — Temporal Graph Network：从静态边到按时间到达的事件流

此前的 GCN、LightGCN、R-GCN 和 HGT 都把图看成某个时刻已经存在的结构。本课中，一条边不再只是 `(source, destination)`，而是按时间到达的事件 `(source, destination, timestamp, message)`。模型预测当前事件之前，只能读取更早的历史。

In [1]:
from pathlib import Path
import sys
import time
import matplotlib.pyplot as plt
import pandas as pd
import torch

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from crosscity.data.temporal_graph import load_jodie_wikipedia
from crosscity.models.temporal_graph import StaticTemporalLinkPredictor, TGNLinkPredictor
from crosscity.training.temporal_graph import train_static_temporal_baseline, train_tgn
from crosscity.utils import seed_everything
if torch.cuda.is_available(): device = torch.device('cuda')
elif torch.backends.mps.is_available(): device = torch.device('mps')
else: device = torch.device('cpu')
print('device:', device)
if device.type == 'cuda': print('GPU:', torch.cuda.get_device_name(0))

/Users/wangyue/STmining/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: mps


## 1. 公开数据 JODIE Wikipedia

数据包含约 15.7 万次用户编辑 Wikipedia 页面的交互，约 9,227 个用户/页面节点。每个事件包含：

- `src`：执行编辑的用户；
- `dst`：被编辑的页面；
- `t`：事件时间戳；
- `msg`：172 维事件特征；
- `y`：原数据附带标签，本课链接预测暂不使用。

第一次运行会下载 `wikipedia.csv`。若远程服务器不能联网，可在本地下载后上传到 `data/raw/jodie_wikipedia/raw/wikipedia.csv`，PyG 随后会自动生成 `processed/data.pt`。

In [2]:
splits = load_jodie_wikipedia(ROOT / 'data/raw/jodie_wikipedia')
data = splits.full
print(data)
print('nodes/events/message dim:', data.num_nodes, data.num_events, data.msg.size(-1))
print('train/validation/test events:',
      splits.train.num_events, splits.validation.num_events, splits.test.num_events)
print('timestamps:', int(data.t.min()), '→', int(data.t.max()))

Processing...
/Users/wangyue/STmining/.venv/lib/python3.12/site-packages/torch_geometric/datasets/jodie.py:104: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  src = torch.from_numpy(df.iloc[:, 0].values).to(torch.long)
Done!


TemporalData(src=[157474], dst=[157474], t=[157474], msg=[157474, 172], y=[157474])
nodes/events/message dim: 9227 157474 172
train/validation/test events: 110232 23621 23621
timestamps: 0 → 2678373


## 2. `TemporalData` 不是快照列表

一种动态图表示法是每小时保存一张图 $G_1,G_2,\ldots$，称为 snapshot。Wikipedia 事件发生时间不规则，更自然的表示是连续事件流：

$$e_k=(u_k,v_k,t_k,x_k),\qquad t_1\le t_2\le\cdots\le t_M$$

其中 $x_k$ 是事件消息。`TemporalData` 直接保存五个等长张量，而不是先构造一张包含全部未来边的 `edge_index`。一条用户—页面边可以重复发生，每次事件时间和消息都不同。

In [ ]:
pd.DataFrame({
    'src': data.src[:8].tolist(), 'dst': data.dst[:8].tolist(),
    'timestamp': data.t[:8].tolist(), 'label': data.y[:8].tolist(),
})

## 3. 时间切分与泄漏

训练、验证、测试必须是连续时间段：

```text
过去 70%                接着 15%          最后 15%
──────── train ────────│── validation ──│──── test ───→ time
```

随机切分会让未来编辑进入训练。batch 也不能 shuffle。验证开始时可以继承全部训练历史，测试开始时可以继承训练和验证历史，因为线上系统在对应时间确实已经观察到这些事件。

最关键的顺序是：**先预测事件，再把该事件写入状态**。若先更新 memory，模型等于在回答问题之前已经看见答案。

In [ ]:
boundaries = pd.DataFrame([
    {'split': name, 'events': part.num_events,
     'start_t': int(part.t.min()), 'end_t': int(part.t.max())}
    for name, part in [('train', splits.train), ('validation', splits.validation), ('test', splits.test)]
])
boundaries

## 4. 我们预测什么？

给定当前用户 $u$ 和截止时间 $t$ 的历史，判断页面 $v$ 是否是其真实交互对象。正例是实际事件 $(u,v,t)$；负例把 destination 换成随机页面 $v^-$。模型输出 logits：

$$s^+=f(z_u(t^-),z_v(t^-)),\qquad s^-=f(z_u(t^-),z_{v^-}(t^-))$$

$t^-$ 强调表示只能包含事件发生前的信息。使用二元交叉熵提高正例分数、降低负例分数。和推荐系统一样，随机负页面是未观察项，不一定是现实中的永久负例。

## 5. 静态 embedding 基线

基线为每个用户和页面学习固定向量 $e_i$，评分只依赖节点 ID：

$$s(u,v)=MLP(e_u,e_v)$$

无论用户昨天还是一年前编辑页面，其 embedding 都一样。它能学习长期偏好和热门页面，却不知道最近发生了什么。保留这个基线可以检验 TGN 的时间状态是否真的有价值。

## 6. TGN 的第一种状态：节点 memory

每个节点维护记忆 $m_i(t)$ 和最后更新时间 $t_i^{last}$。新事件 $(u,v,t,x)$ 先产生消息：

$$message_u=[m_u\Vert m_v\Vert x\Vert \phi(t-t_u^{last})]$$

若同一节点积累多条待处理消息，需要聚合。本课使用 `LastAggregator`，只保留时间最新的消息，再通过 GRU 更新：

$$m_u(t)=GRU(message_u,m_u(t^-))$$

memory 不是普通模型参数：参数通过梯度学习，而 memory 是按事件顺序运行时改变的节点状态。每个训练 epoch 开始都必须清空，否则下一轮会继承上一轮的未来历史。

## 7. 时间编码器如何表示时间间隔？

原始时间戳很大，真正相关的通常是间隔 $\Delta t=t-t_i^{last}$。PyG 时间编码器学习不同频率：

$$\phi(\Delta t)=cos(W\Delta t+b)$$

余弦让多个维度描述不同时间尺度：某些维度可以对短期变化敏感，另一些维度变化更慢。它不是把 timestamp 当类别，也不意味着真实过程一定周期性；频率和相位由任务损失学习。

In [ ]:
demo_model = TGNLinkPredictor(data.num_nodes, data.msg.size(-1),
                              memory_dim=16, time_dim=8, embedding_dim=16)
delta_t = torch.tensor([0., 1., 10., 100.])
encoded = demo_model.memory.time_enc(delta_t).detach()
print('time input shape:', tuple(delta_t.shape))
print('encoded shape:', tuple(encoded.shape))
encoded

## 8. TGN 的第二种状态：最近邻居缓存

memory 压缩节点经历过什么，但不直接保存局部拓扑。`LastNeighborLoader(size=10)` 为每个节点保留最近 10 个交互邻居及其事件编号。预测时只取当前 batch 涉及节点的历史子图。

两种状态的分工：

- memory：历史内容摘要；
- neighbor cache：最近连接结构和事件索引。

缓存必须和 memory 一起按时间更新、每个 epoch 一起清空。`size` 越大，历史上下文更丰富，但计算和噪声也增加。

## 9. 历史邻居注意力

缓存取出历史边后，`TransformerConv` 根据节点 memory 和边属性做注意力。边属性由事件消息与相对时间拼接：

$$edge\_attr_{j\to i}=[x_{ji}\Vert\phi(t_i^{last}-t_{ji})]$$

注意力决定不同历史邻居对当前表示的重要程度：

$$z_i(t)=TransformerConv(m_i,\mathcal N_i^{recent},edge\_attr)$$

因此 TGN 不是单纯的 RNN，也不是在完整静态图上跑 Transformer；它把节点记忆、最近历史子图和时间差编码组合起来。

## 10. 一个 batch 的严格操作顺序

```text
1. 读取当前事件节点
2. 查询事件之前的最近邻居
3. 读取旧 memory 和 last_update
4. 生成时间感知 embedding
5. 对正/负链接打分并计算 loss
6. 将真实事件写入 memory 和邻居缓存
7. 反向传播，detach memory
```

batch 内事件并行预测，所以同一 batch 后面的事件看不到该 batch 前面的事件。这比逐事件更新更快，也避免在并行预测中偷看当前 batch；代价是时间分辨率由 batch size 决定。

## 11. AP 与 AUC

AUC 表示随机取一条正事件和一条负事件时，正例分数更高的概率。AP 汇总 precision-recall 曲线，更关注正例排名质量。当前评价每个正例采一个负 destination，因此二者都能使用，但数值会受负采样协议影响，不能与使用全量候选或 100 个负例的论文直接横比。

In [ ]:
RUN_FULL = False
seeds = [42, 43, 44] if RUN_FULL else [42]
epochs = 30 if RUN_FULL else 10
batch_size = 200
rows, histories = [], {}
for seed in seeds:
    for name in ('StaticEmbedding', 'TGN'):
        seed_everything(seed)
        if device.type == 'cuda': torch.cuda.reset_peak_memory_stats()
        started = time.perf_counter()
        if name == 'StaticEmbedding':
            model = StaticTemporalLinkPredictor(data.num_nodes, embedding_dim=100)
            result = train_static_temporal_baseline(
                model, splits, device=device, batch_size=batch_size, max_epochs=epochs)
        else:
            model = TGNLinkPredictor(
                data.num_nodes, data.msg.size(-1), memory_dim=100,
                time_dim=100, embedding_dim=100)
            result = train_tgn(
                model, splits, device=device, batch_size=batch_size,
                neighbor_size=10, max_epochs=epochs)
        elapsed = time.perf_counter() - started
        peak_mb = torch.cuda.max_memory_allocated() / 1024**2 if device.type == 'cuda' else float('nan')
        rows.append({
            'seed': seed, 'model': name, 'best_epoch': result.best_epoch,
            'validation_ap': result.validation.average_precision,
            'validation_auc': result.validation.auc,
            'test_ap': result.test.average_precision, 'test_auc': result.test.auc,
            'seconds': elapsed, 'peak_cuda_mb': peak_mb,
        })
        histories[(seed, name)] = result.history
results = pd.DataFrame(rows)
display(results.sort_values('validation_ap', ascending=False))
results.groupby('model')[['validation_ap', 'test_ap', 'test_auc', 'seconds', 'peak_cuda_mb']].agg(['mean', 'std'])

In [ ]:
for (seed, name), history in histories.items():
    if seed == seeds[0]:
        frame = pd.DataFrame(history)
        plt.plot(frame.epoch, frame.validation_ap, label=name)
plt.xlabel('epoch'); plt.ylabel('validation AP')
plt.title('Wikipedia temporal link prediction')
plt.legend(); plt.show()

## 12. 如何解释结果

- `TGN > StaticEmbedding`：事件顺序、时间间隔或近期邻居提供了固定 ID 偏好之外的信息。
- `TGN ≈ StaticEmbedding`：长期身份偏好已经很强，或当前负采样太容易。
- TGN 后期 validation 下降：仍然可能过拟合；有 memory 不代表不会过拟合。
- batch size 改变结果：它同时改变优化 batch 和状态可见性的时间粒度，不是纯粹性能参数。
- neighbor size 增大不一定更好：更旧邻居可能只是噪声。

代码只用 validation 选择最佳 epoch。选择后加载 checkpoint，清空状态，再严格回放 train→validation，最后只评一次 test。

## 13. 与 STGCN 的联系和区别

| | Wikipedia TGN | 交通 STGCN |
|---|---|---|
| 时间 | 不规则事件 | 固定 5 分钟采样 |
| 图 | 交互发生时动态出现 | 道路/距离邻接通常固定 |
| 状态 | 节点 memory 按事件更新 | 时间卷积读取固定窗口 |
| 目标 | 下一条链接 | 未来连续速度 |

两者都要求只能用过去预测未来，但不能简单把一种架构搬到另一种数据上。TGN 适合事件驱动动态图；STGCN 适合规则网格时间序列。之后可以学习动态图快照、Temporal Transformer，以及二者如何与空间关系结合。

## 本课结论

1. 时间图的基本单位是有序事件，而不只是带 timestamp 的静态边。
2. 时间切分、batch 顺序和先预测后更新共同防止未来泄漏。
3. memory 保存历史内容状态，neighbor cache 保存近期结构状态。
4. 时间编码器学习不同时间尺度，历史注意力选择相关邻居。
5. TGN 的状态必须在 epoch 开头清空，在 validation/test 边界连续继承。
6. 模型复杂度仍需由静态基线、多 seed、资源消耗和严格 test 协议共同检验。